# Smart MCQ Solver Challenge - Milestone 2
This notebook computes all answers required for the milestone. Run all cells on Kaggle with the competition dataset.

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import numpy as np

dataset=load_dataset('csv',data_files={'train':'/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'})['train']
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


## Q1 Combined text

In [2]:
dataset=dataset.map(lambda x:{'combined_text':x['prompt']+' '+x['A']})
print('Answer Q1:',len(dataset[51]['combined_text']))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Answer Q1: 614


## Q2-Q4 Tokenizer

In [4]:
tokenizer=AutoTokenizer.from_pretrained('bert-base-uncased')
print('Q2 Vocabulary Size:',tokenizer.vocab_size)
print('Q3 SEP Token ID:',tokenizer.sep_token_id)
prompts = list(dataset["prompt"])

enc = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Q4 input_ids shape:", enc["input_ids"].shape)
print('Q5 Head Dimension:',768//12)

Q2 Vocabulary Size: 30522
Q3 SEP Token ID: 102
Q4 input_ids shape: torch.Size([2000, 128])
Q5 Head Dimension: 64


## Q6-Q8 BERT

In [5]:
model=AutoModel.from_pretrained('bert-base-uncased')
inputs=tokenizer(dataset[0]['prompt'],return_tensors='pt')
outputs=model(**inputs)
print('Q6 last_hidden_state shape:',outputs.last_hidden_state.shape)
cls=outputs.last_hidden_state[0,0]
print('Q7 sum first5:',round(cls[:5].sum().item(),4))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6 last_hidden_state shape: torch.Size([1, 31, 768])
Q7 sum first5: -1.2001


## Q8 Attention

In [6]:
model_att=AutoModel.from_pretrained('bert-base-uncased',output_attentions=True)
text='Light-ion fusion is a technique.'
inp=tokenizer(text,return_tensors='pt')
out=model_att(**inp)
tokens=tokenizer.convert_ids_to_tokens(inp['input_ids'][0])
fusion_idx=tokens.index('fusion') if 'fusion' in tokens else next(i for i,t in enumerate(tokens) if 'fusion' in t)
att=out.attentions[-1][0,0]
print(tokens)
print('Q8 attention:',round(att[0,fusion_idx].item(),4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Q8 attention: 0.1025


## Q9 Context-aware Embeddings

In [7]:
st=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb=st.encode([dataset[0]['prompt'],dataset[0]['B']],convert_to_tensor=True)
print('Q9 similarity:',round(util.cos_sim(emb[0],emb[1]).item(),4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9 similarity: 0.7658


## Q10 Ranking Pipelines

In [8]:
labels=['A','B','C','D','E']

def map3(preds,truth):
    s=0
    for p,t in zip(preds,truth):
        if t in p:
            s+=1/(p.index(t)+1)
    return s/len(truth)

tfidf_preds=[]
mini_preds=[]
for row in dataset:
    opts=[row[x] for x in labels]
    docs=[row['prompt']]+opts
    X=TfidfVectorizer().fit_transform(docs)
    sims=cosine_similarity(X[0:1],X[1:]).flatten()
    order=np.argsort(-sims)
    tfidf_preds.append([labels[i] for i in order[:3]])

emb=st.encode([r['prompt'] for r in dataset]+[r[l] for r in dataset for l in labels],convert_to_numpy=True)

N=len(dataset)
for i,row in enumerate(dataset):
    p=emb[i]
    opts=emb[N+i*5:N+(i+1)*5]
    sims=util.cos_sim(torch.tensor(p),torch.tensor(opts)).numpy().flatten()
    order=np.argsort(-sims)
    mini_preds.append([labels[j] for j in order[:3]])

truth=list(dataset['answer'])
print('MiniLM MAP@3:',map3(mini_preds,truth))
count=0
for t,a,b in zip(truth,tfidf_preds,mini_preds):
    if t not in a and t in b:
        count+=1
print('Improvement count:',count)

MiniLM MAP@3: 0.42308333333333487
Improvement count: 522


## Q11-Q12 Zero-shot

In [9]:
zs=pipeline('zero-shot-classification',model='facebook/bart-large-mnli')
row=dataset[1]
cand=[row['A'],row['B'],row['C']]
r1=zs(row['prompt'],cand)
print('Q11 top probability:',round(r1['scores'][0],4))
r2=zs(row['prompt'],cand,multi_label=True)
print('Q12 difference:',abs(sum(r1['scores'])-sum(r2['scores'])))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11 top probability: 0.4575
Q12 difference: 0.9994903629012697


## Q13 FLAN-T5

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

r = dataset[0]

prompt = (
    f"Question: {r['prompt']}. "
    f"Is the correct answer A: {r['A']} or B: {r['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=5)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Q13 output:", answer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 output: B
